# Humanitarian News Classification Notebook

The Jupyter notebook pipeline demonstrates a pragmatic approach to collecting and structuring humanitarian news data. It balances simplicity, reproducibility, and relevance to MSF operations while remaining transparent about its assumptions and limitations.

## Key Sections
1. **Data Processing**: Loading, analysing and preprocessing the dataset for classification models.
2. **Topic Modelling**: Applying classification model for topic modelling.
3. **Sentiment Classification**: Applying classification model for sentiment analysis.
4. **Output**: Generating structured output.

Repository: [JP-Thoma/Humanitarian-News-Classification](https://github.com/JP-Thoma/Humanitarian-News-Classification)

## 1. Data Processing: Loading, analysing and preprocessing the dataset for classification models

In [1]:
# installing dependencies
!pip install -r requirements.txt


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [60]:
# load libraries
import os
from dotenv import load_dotenv
from newsapi import NewsApiClient
import pandas as pd
from transformers import pipeline

In [61]:
# load api key and HuggingFace login
load_dotenv()

API_KEY = os.getenv("NEWSAPI_KEY")

if API_KEY is None:
    raise ValueError("API key not found. Please check your .env file.")

In [63]:
# intitialise NewsAPI Client
newsapi = NewsApiClient(api_key=API_KEY)

In [64]:
# specify keywords
QUERY = """
(
  outbreak OR epidemic OR pandemic OR cholera OR measles OR malaria OR dengue OR tuberculosis OR Ebola OR diphtheria OR malnutrition OR starvation OR famine OR "maternal mortality" OR trauma OR "mass casualty" OR "burn injuries" 
  OR "health system" OR "hospital overwhelmed"
)
AND
(
  war OR conflict OR violence OR bombardment OR airstrike OR siege OR displacement OR refugee OR IDP OR famine OR "food insecurity"
)
AND
(
  humanitarian OR NGO OR "aid workers" OR "medical charity"
)
"""

In [65]:
# specify country selection
MSF_COUNTRIES = [
    # Africa
    "Benin", "Burkina Faso", "Burundi", "Cameroon", "Central African Republic",
    "Chad", "Comoros", "Democratic Republic of Congo", "Eswatini", "Ethiopia",
    "Guinea", "Guinea-Bissau", "Ivory Coast", "Kenya", "Liberia",
    "Madagascar", "Malawi", "Mali", "Mauritania", "Mozambique",
    "Niger", "Nigeria", "Sierra Leone", "Somalia", "Somaliland",
    "South Africa", "South Sudan", "Sudan", "Tanzania", "Uganda", "Zimbabwe",

    # Middle East & North Africa
    "Egypt", "Iran", "Iraq", "Jordan", "Lebanon",
    "Libya", "Palestine", "Syria", "Yemen",

    # Asia & Pacific
    "Bangladesh", "Cambodia", "North Korea", "Hong Kong", "India",
    "Indonesia", "Kiribati", "Malaysia", "Myanmar", "Pakistan",
    "Papua New Guinea", "Philippines", "Thailand",

    # Europe & Central Asia
    "Afghanistan", "Armenia", "Azerbaijan", "Belarus", "Belgium",
    "Bulgaria", "France", "Greece", "Italy", "Kazakhstan",
    "Kyrgyzstan", "Latvia", "Lithuania", "Poland",
    "Russia", "Serbia", "Tajikistan", "Turkey",
    "Ukraine", "United Kingdom", "Uzbekistan",

    # Americas
    "Brazil", "Colombia", "El Salvador", "Guatemala",
    "Haiti", "Honduras", "Jamaica", "Mexico",
    "Nicaragua", "Panama", "United States", "Venezuela"
]

In [66]:
# fetch articles
articles_response = newsapi.get_everything(
    q=QUERY,
    from_param="2026-04-15",
    to="2026-04-25",
    language="en",
    sort_by="publishedAt",
    page_size=100
)

print(f"Total results: {articles_response['totalResults']}")
print(f"Articles retrieved: {len(articles_response['articles'])}")

articles_response['articles'][0]

Total results: 182
Articles retrieved: 92


{'source': {'id': 'nbc-news', 'name': 'NBC News'},
 'author': 'Matt Bradley',
 'title': 'Disease, hunger and Israeli strikes: Six months after Trump’s ceasefire in Gaza',
 'description': 'She spent the winter struggling to protect her six young children from the biting cold and driving rain.',
 'url': 'https://www.nbcnews.com/world/gaza/disease-hunger-israeli-strikes-six-months-trumps-ceasefire-gaza-rcna341013',
 'urlToImage': 'https://media-cldnry.s-nbcnews.com/image/upload/t_nbcnews-fp-1200-630,f_auto,q_auto:best/rockcms/2026-04/260422-khan-younis-mb-1739-141710.jpg',
 'publishedAt': '2026-04-25T10:08:32Z',
 'content': 'Optimistic plans to improve the enclaves security, provide reconstruction and humanitarian relief, and institute a more permanent governance structure in Gaza are gridlocked by diplomatic disagreemen… [+3323 chars]'}

In [67]:
# create dataframe
articles = articles_response['articles']

df = pd.json_normalize(articles)
df.head()

,author,title,description,url,urlToImage,publishedAt,content,source.id,source.name
0,Matt Bradley,"Disease, hunger and Israeli strikes: Six month...",She spent the winter struggling to protect her...,https://www.nbcnews.com/world/gaza/disease-hun...,https://media-cldnry.s-nbcnews.com/image/uploa...,2026-04-25T10:08:32Z,Optimistic plans to improve the enclaves secur...,nbc-news,NBC News
1,Pache Chiedozie,Biafra: 'Nigeria's breakup likely inevitable a...,"A former mayor in the United States,, Mike Arn...",https://dailypost.ng/2026/04/25/biafra-nigeria...,https://dailypost.ng/wp-content/uploads/2025/1...,2026-04-25T09:35:32Z,"A former mayor in the United States,, Mike Arn...",NaN,Daily Post Nigeria
2,Oboh,Sachet Alcohol: Sacrificing our children for p...,NAFDAC’s decision to ban the production and sa...,https://www.vanguardngr.com/2026/04/sachet-alc...,https://cdn.vanguardngr.com/wp-content/uploads...,2026-04-25T06:46:44Z,By Magnus Onyibe\r\nThe National Agency For Fo...,NaN,Vanguard
3,Sally Rooney,Sally Rooney: Fragility of global fuel systems...,We cannot increase the amount of oil and gas a...,https://www.irishtimes.com/life-style/people/2...,https://www.irishtimes.com/resizer/v2/PJVLHBAR...,2026-04-25T05:00:00Z,"When goods are scarce, how are they distribute...",the-irish-times,The Irish Times
4,Víctor M. Vázquez Rodríguez,Structural Consequences of Economic Crisis and...,"Since 2025, Puerto Rico has become a front lin...",https://smallwarsjournal.com/2026/04/24/struct...,https://smallwarsjournal.com/wp-content/upload...,2026-04-24T23:00:33Z,"Since 2025, Puerto Rico has become a front lin...",NaN,Smallwarsjournal.com


In [68]:
# select relevant columns
df = df[[
    "source.name",
    "author",
    "title",
    "description",
    "url",
    "publishedAt",
    "content"
]]

df.rename(columns={
    "source.name": "source"
}, inplace=True)

print(f"Final dataset size: {len(df)}")
df.head()

Final dataset size: 92


,source,author,title,description,url,publishedAt,content
0,NBC News,Matt Bradley,"Disease, hunger and Israeli strikes: Six month...",She spent the winter struggling to protect her...,https://www.nbcnews.com/world/gaza/disease-hun...,2026-04-25T10:08:32Z,Optimistic plans to improve the enclaves secur...
1,Daily Post Nigeria,Pache Chiedozie,Biafra: 'Nigeria's breakup likely inevitable a...,"A former mayor in the United States,, Mike Arn...",https://dailypost.ng/2026/04/25/biafra-nigeria...,2026-04-25T09:35:32Z,"A former mayor in the United States,, Mike Arn..."
2,Vanguard,Oboh,Sachet Alcohol: Sacrificing our children for p...,NAFDAC’s decision to ban the production and sa...,https://www.vanguardngr.com/2026/04/sachet-alc...,2026-04-25T06:46:44Z,By Magnus Onyibe\r\nThe National Agency For Fo...
3,The Irish Times,Sally Rooney,Sally Rooney: Fragility of global fuel systems...,We cannot increase the amount of oil and gas a...,https://www.irishtimes.com/life-style/people/2...,2026-04-25T05:00:00Z,"When goods are scarce, how are they distribute..."
4,Smallwarsjournal.com,Víctor M. Vázquez Rodríguez,Structural Consequences of Economic Crisis and...,"Since 2025, Puerto Rico has become a front lin...",https://smallwarsjournal.com/2026/04/24/struct...,2026-04-24T23:00:33Z,"Since 2025, Puerto Rico has become a front lin..."


In [69]:
# light cleaning
# Drop duplicates based on title
df.drop_duplicates(subset="title", inplace=True)

# Drop rows with no description (important for later classification)
df = df[df["description"].notna()]

df.reset_index(drop=True, inplace=True)

print(f"Final dataset size: {len(df)}")

Final dataset size: 91


In [70]:
# add country column 
def extract_country(text):
    text = text.lower()
    for country in MSF_COUNTRIES:
        if country.lower() in text:
            return country
    return "Unknown"

df["country"] = df["content"].apply(extract_country)

df.head()

,source,author,title,description,url,publishedAt,content,country
0,NBC News,Matt Bradley,"Disease, hunger and Israeli strikes: Six month...",She spent the winter struggling to protect her...,https://www.nbcnews.com/world/gaza/disease-hun...,2026-04-25T10:08:32Z,Optimistic plans to improve the enclaves secur...,Unknown
1,Daily Post Nigeria,Pache Chiedozie,Biafra: 'Nigeria's breakup likely inevitable a...,"A former mayor in the United States,, Mike Arn...",https://dailypost.ng/2026/04/25/biafra-nigeria...,2026-04-25T09:35:32Z,"A former mayor in the United States,, Mike Arn...",Niger
2,Vanguard,Oboh,Sachet Alcohol: Sacrificing our children for p...,NAFDAC’s decision to ban the production and sa...,https://www.vanguardngr.com/2026/04/sachet-alc...,2026-04-25T06:46:44Z,By Magnus Onyibe\r\nThe National Agency For Fo...,Unknown
3,The Irish Times,Sally Rooney,Sally Rooney: Fragility of global fuel systems...,We cannot increase the amount of oil and gas a...,https://www.irishtimes.com/life-style/people/2...,2026-04-25T05:00:00Z,"When goods are scarce, how are they distribute...",Unknown
4,Smallwarsjournal.com,Víctor M. Vázquez Rodríguez,Structural Consequences of Economic Crisis and...,"Since 2025, Puerto Rico has become a front lin...",https://smallwarsjournal.com/2026/04/24/struct...,2026-04-24T23:00:33Z,"Since 2025, Puerto Rico has become a front lin...",Unknown


In [71]:
# check country frequency
df["country"].value_counts()

country
Unknown                         48
Sudan                           10
Iran                             6
Democratic Republic of Congo     5
Niger                            4
Lebanon                          3
Mali                             2
Ethiopia                         2
North Korea                      2
Italy                            1
France                           1
Kazakhstan                       1
Libya                            1
Liberia                          1
Zimbabwe                         1
Yemen                            1
Venezuela                        1
Palestine                        1
Name: count, dtype: int64

## 2. Topic Modelling: Applying classification model for topic modelling

In [72]:
# load topic modelling pipeline from Hugging Face
from transformers import pipeline
pipe_topics = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 4391.70it/s]


In [73]:
labels = [
    "Conflict-Driven Crisis (war, violence, attacks on healthcare)",
    "Epidemic & Disease-Driven Crisis (cholera, outbreak, infectious disease)",
    "Food & Nutrition-Driven Crisis (famine, malnutrition, starvation)",
    "Displacement-Driven Crisis (refugees, camps, migration)",
    "Environmental Disaster-Driven Crisis (flood, drought, earthquake)"
]

In [74]:
# test
pipe_topics("'WASHINGTON, D.C. As the U.S. House of Representatives nears a crucial vote on the farm bill, Catholic Relief Services (CRS) is urging lawmakers not to sideline international hunger relief",
            labels)

{'sequence': "'WASHINGTON, D.C. As the U.S. House of Representatives nears a crucial vote on the farm bill, Catholic Relief Services (CRS) is urging lawmakers not to sideline international hunger relief",
 'labels': ['Food & Nutrition-Driven Crisis (famine, malnutrition, starvation)',
  'Displacement-Driven Crisis (refugees, camps, migration)',
  'Epidemic & Disease-Driven Crisis (cholera, outbreak, infectious disease)',
  'Conflict-Driven Crisis (war, violence, attacks on healthcare)',
  'Environmental Disaster-Driven Crisis (flood, drought, earthquake)'],
 'scores': [0.7669042348861694,
  0.08863211423158646,
  0.07402593642473221,
  0.040508419275283813,
  0.029929213225841522]}

In [75]:
def classify_text(text):
    """classify news content into provided labels"""
    if not isinstance(text, str):
        return "Unknown"
    
    result = pipe_topics(text, labels)
    return result["labels"][0]

df["category"] = df["description"].apply(classify_text)

df.head()

,source,author,title,description,url,publishedAt,content,country,category
0,NBC News,Matt Bradley,"Disease, hunger and Israeli strikes: Six month...",She spent the winter struggling to protect her...,https://www.nbcnews.com/world/gaza/disease-hun...,2026-04-25T10:08:32Z,Optimistic plans to improve the enclaves secur...,Unknown,"Displacement-Driven Crisis (refugees, camps, m..."
1,Daily Post Nigeria,Pache Chiedozie,Biafra: 'Nigeria's breakup likely inevitable a...,"A former mayor in the United States,, Mike Arn...",https://dailypost.ng/2026/04/25/biafra-nigeria...,2026-04-25T09:35:32Z,"A former mayor in the United States,, Mike Arn...",Niger,"Conflict-Driven Crisis (war, violence, attacks..."
2,Vanguard,Oboh,Sachet Alcohol: Sacrificing our children for p...,NAFDAC’s decision to ban the production and sa...,https://www.vanguardngr.com/2026/04/sachet-alc...,2026-04-25T06:46:44Z,By Magnus Onyibe\r\nThe National Agency For Fo...,Unknown,"Displacement-Driven Crisis (refugees, camps, m..."
3,The Irish Times,Sally Rooney,Sally Rooney: Fragility of global fuel systems...,We cannot increase the amount of oil and gas a...,https://www.irishtimes.com/life-style/people/2...,2026-04-25T05:00:00Z,"When goods are scarce, how are they distribute...",Unknown,"Displacement-Driven Crisis (refugees, camps, m..."
4,Smallwarsjournal.com,Víctor M. Vázquez Rodríguez,Structural Consequences of Economic Crisis and...,"Since 2025, Puerto Rico has become a front lin...",https://smallwarsjournal.com/2026/04/24/struct...,2026-04-24T23:00:33Z,"Since 2025, Puerto Rico has become a front lin...",Unknown,"Displacement-Driven Crisis (refugees, camps, m..."


In [76]:
# check category frequency
df["category"].value_counts()

category
Conflict-Driven Crisis (war, violence, attacks on healthcare)               43
Displacement-Driven Crisis (refugees, camps, migration)                     24
Epidemic & Disease-Driven Crisis (cholera, outbreak, infectious disease)    14
Food & Nutrition-Driven Crisis (famine, malnutrition, starvation)            9
Environmental Disaster-Driven Crisis (flood, drought, earthquake)            1
Name: count, dtype: int64

## 3. Sentiment Classification: Applying classification model for sentiment analysis

In [77]:
# load sentiment analysis pipeline from Hugging Face
pipe_sentiments = pipeline("text-classification", model="cardiffnlp/twitter-roberta-base-sentiment-latest")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 30966.21it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [78]:
#Test
pipe_sentiments("WASHINGTON, D.C. As the U.S. House of Representatives nears a crucial vote on the farm bill, Catholic Relief Services (CRS) is urging lawmakers not to sideline international hunger relief")

[{'label': 'neutral', 'score': 0.8960832357406616}]

In [79]:
def sentiment_text(text):
    """Classify news text into Positive, Neutral, Negative"""
    
    if not isinstance(text, str):
        return "Unknown"
    
    result = pipe_sentiments(text)[0]
    
    label = result["label"]

    return label

df["sentiment"] = df["description"].apply(sentiment_text)

df.head()

,source,author,title,description,url,publishedAt,content,country,category,sentiment
0,NBC News,Matt Bradley,"Disease, hunger and Israeli strikes: Six month...",She spent the winter struggling to protect her...,https://www.nbcnews.com/world/gaza/disease-hun...,2026-04-25T10:08:32Z,Optimistic plans to improve the enclaves secur...,Unknown,"Displacement-Driven Crisis (refugees, camps, m...",negative
1,Daily Post Nigeria,Pache Chiedozie,Biafra: 'Nigeria's breakup likely inevitable a...,"A former mayor in the United States,, Mike Arn...",https://dailypost.ng/2026/04/25/biafra-nigeria...,2026-04-25T09:35:32Z,"A former mayor in the United States,, Mike Arn...",Niger,"Conflict-Driven Crisis (war, violence, attacks...",neutral
2,Vanguard,Oboh,Sachet Alcohol: Sacrificing our children for p...,NAFDAC’s decision to ban the production and sa...,https://www.vanguardngr.com/2026/04/sachet-alc...,2026-04-25T06:46:44Z,By Magnus Onyibe\r\nThe National Agency For Fo...,Unknown,"Displacement-Driven Crisis (refugees, camps, m...",neutral
3,The Irish Times,Sally Rooney,Sally Rooney: Fragility of global fuel systems...,We cannot increase the amount of oil and gas a...,https://www.irishtimes.com/life-style/people/2...,2026-04-25T05:00:00Z,"When goods are scarce, how are they distribute...",Unknown,"Displacement-Driven Crisis (refugees, camps, m...",neutral
4,Smallwarsjournal.com,Víctor M. Vázquez Rodríguez,Structural Consequences of Economic Crisis and...,"Since 2025, Puerto Rico has become a front lin...",https://smallwarsjournal.com/2026/04/24/struct...,2026-04-24T23:00:33Z,"Since 2025, Puerto Rico has become a front lin...",Unknown,"Displacement-Driven Crisis (refugees, camps, m...",neutral


In [80]:
# check sentiment frequency
df["sentiment"].value_counts()

sentiment
negative    50
neutral     34
positive     7
Name: count, dtype: int64

## 4. Output: Generating structured output

In [81]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 91 entries, 0 to 90
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   source       91 non-null     str  
 1   author       87 non-null     str  
 2   title        91 non-null     str  
 3   description  91 non-null     str  
 4   url          91 non-null     str  
 5   publishedAt  91 non-null     str  
 6   content      91 non-null     str  
 7   country      91 non-null     str  
 8   category     91 non-null     str  
 9   sentiment    91 non-null     str  
dtypes: str(10)
memory usage: 7.2 KB


In [82]:
df.tail()

,source,author,title,description,url,publishedAt,content,country,category,sentiment
86,Theeconomiccollapseblog.com,Michael,The U.S. Navy Just Blew A Giant Hole In An Ira...,Apparently the Trump administration was not ki...,http://theeconomiccollapseblog.com/the-u-s-nav...,2026-04-19T20:48:03Z,Apparently the Trump administration was not ki...,Iran,"Conflict-Driven Crisis (war, violence, attacks...",negative
87,Antiwar.com,J.D. Hester,Remembering the Catholic Bishop Behind Lebanon...,Following President Donald Trump’s vicious att...,https://www.antiwar.com/blog/2026/04/19/rememb...,2026-04-19T15:41:35Z,Following President Donald Trumps vicious atta...,Iran,"Conflict-Driven Crisis (war, violence, attacks...",neutral
88,Christianity.com,Dr. Dennis Sempebwa,Facing the Reality of War with Biblical Truth,"Explore a biblical perspective on war, urging ...",https://www.christianity.com/wiki/current-even...,2026-04-19T04:00:00Z,The church must pursue peace without surrender...,Unknown,"Displacement-Driven Crisis (refugees, camps, m...",neutral
89,Activistpost.com,Editor,When War Teaches Medicine,War is the most unrestrained expression of hum...,https://www.activistpost.com/when-war-teaches-...,2026-04-19T01:05:00Z,War is the most unrestrained expression of hum...,Unknown,"Conflict-Driven Crisis (war, violence, attacks...",negative
90,PBS,"Isabel DeBre, Associated Press",Witnesses detail deadly Israeli strikes on med...,"The three strikes, which killed four paramedic...",https://www.pbs.org/newshour/world/witnesses-d...,2026-04-18T20:55:00Z,"NABATIYEH, Lebanon (AP) It was late morning wh...",Lebanon,"Conflict-Driven Crisis (war, violence, attacks...",negative


In [85]:
df.to_csv("humanitarian_news_classification.csv", sep=";", index=False, encoding="utf-8-sig")
#df.to_json("humanitarian_news_classification.json", orient="records", indent=2)